# Create .csv table with DLC results

In [1]:
##### LOCAL #####

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import os
from pathlib import Path
import re
from scipy import stats
from scipy.ndimage import median_filter
from scipy.interpolate import interp1d
from scipy.spatial import distance
import openpyxl


In [2]:
##############################################################################
# Choose the folder containing the .h5 files
##############################################################################
folder_path = "//10.69.168.1/crnldata/forgetting/Carla/Cheeseboard/"
counter = 0
counterProbe = 0
day = 1
trial = 1
previousmice = 0
previous_session_time = 0
previous_session_type = None
Summary_table = pd.DataFrame()
Sholl_table = pd.DataFrame()


In [3]:
##############################################################################
# Define functions
##############################################################################

def calculate_relative_distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def calculate_distance_run(x_coords, y_coords):
    distances = np.sqrt(np.diff(x_coords) ** 2 + np.diff(y_coords) ** 2)
    for i in range(1, len(distances) - 1):
        if np.isnan(distances[i]):
            neighbors = [distances[i-1], distances[i+1]]
            distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
    total_distance_cm = np.nansum(distances) / pixel_to_cm
    return total_distance_cm, distances

def find_long_non_nan_sequences(arr, min_length=5):
    mask = ~np.isnan(arr)  # True for non-NaN values
    diff = np.diff(np.concatenate(([0], mask.astype(int), [0])))  # Add padding to detect edges
    starts = np.where(diff == 1)[0]  # Where a sequence starts
    ends = np.where(diff == -1)[0]   # Where a sequence ends
    sequences = [arr[start:end] for start, end in zip(starts, ends) if (end - start) > min_length]
    return sequences

def remove_outliers_median_filter(data, window=1):
    data = np.array(data, dtype=float)
    filtered_data = np.copy(data)
    half_window = window // 2
    for i in range(len(data)):
        start = max(0, i - half_window)
        end = min(len(data), i + half_window + 1)
        local_values = data[start:end]
        if np.all(np.isnan(local_values)):
            median_value = np.nan
        else:
            median_value = np.nanmedian(local_values)
        if not np.isnan(data[i]):
            filtered_data[i] = median_value
    return filtered_data

def replace_high_speed_points_with_nan(x, y, speed_threshold):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    dx = np.diff(x)
    dy = np.diff(y)
    speeds = np.sqrt(dx**2 + dy**2)
    high_speed_mask = speeds > speed_threshold
    x_out = x.copy()
    y_out = y.copy()
    for i in range(len(high_speed_mask)):
        if high_speed_mask[i]:
            if i > 0 and i < len(x) - 1:
                if speeds[i] > speeds[i - 1]:
                    x_out[i + 1] = np.nan
                    y_out[i + 1] = np.nan
                else:
                    x_out[i] = np.nan
                    y_out[i] = np.nan
    return x_out, y_out

def interpolate_2d_path(x, y, kind='linear', fill='extrapolate'):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    indices = np.arange(len(x))
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(valid_mask) < 2:
        raise ValueError("Not enough valid points to interpolate/extrapolate.")
    interp_x = interp1d(indices[valid_mask], x[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    interp_y = interp1d(indices[valid_mask], y[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    x_filled = x.copy()
    y_filled = y.copy()
    nan_mask = np.isnan(x) | np.isnan(y)
    x_filled[nan_mask] = interp_x(indices[nan_mask])
    y_filled[nan_mask] = interp_y(indices[nan_mask])
    return x_filled, y_filled

def limit_speed(x, y, max_speed):
    dx = np.diff(x.copy())
    dy = np.diff(y.copy())
    speeds = np.sqrt(dx**2 + dy**2)
    for i, t in enumerate(speeds):
        if t > max_speed:
            x[i+1] = x[i]
            y[i+1] = y[i]
            if i+2 < len(x):
                x[i+2] = x[i]
                y[i+2] = y[i]
    return x, y

def remove_short_sequences(arr, max_len=10):
    arr = np.array(arr, dtype='float')
    result = arr.copy()
    is_value = ~np.isnan(arr)
    i = 0
    while i < len(arr):
        if is_value[i]:
            start = i
            while i < len(arr) and is_value[i]:
                i += 1
            end = i
            seq_len = end - start
            if seq_len <= max_len:
                left_nan = (start == 0) or np.isnan(arr[start - 1])
                right_nan = (end == len(arr)) or np.isnan(arr[end])
                if left_nan and right_nan:
                    result[start:end] = np.nan
        else:
            i += 1
    return result

def apply_circular_mask(x, y, x_mask, y_mask, mask_radius):
    path_coords = np.column_stack((x, y))
    mask_center = np.array([x_mask, y_mask])
    dist_to_mask_center = distance.cdist(path_coords, [mask_center]).flatten()
    mask = dist_to_mask_center <= mask_radius
    return x[mask], y[mask]

def get_quadrant(px, py, cx, cy):
    """Return the quadrant (North/South/East/West) of a point relative to a center."""
    dx = px - cx
    dy = py - cy
    diag1 = dx - dy
    diag2 = dx + dy
    if diag1 > 0 and diag2 > 0:
        return "South"
    elif diag1 < 0 and diag2 > 0:
        return "West"
    elif diag1 < 0 and diag2 < 0:
        return "North"
    else:
        return "East"

def compute_performance_metrics(
    individual_x, individual_y,
    start_frame, t, frame_rate,
    reward_x, reward_y, reward_zone,
    table_radius, table_center_x, table_center_y,
    x_start, y_start, start_zone,
    total_distance,
    calculate_relative_distance,
    n_permutations=1000,
    seed=42
):
    individual_x_trim = individual_x[start_frame:int(start_frame + (t * frame_rate))]
    individual_y_trim = individual_y[start_frame:int(start_frame + (t * frame_rate))]

    # Filter NaN pairs jointly to keep x/y synchronized
    mask = ~np.isnan(individual_x_trim) & ~np.isnan(individual_y_trim)
    individual_x_filt = individual_x_trim[mask]
    individual_y_filt = individual_y_trim[mask]

    # ========== Compute Observed metrics ==========
    min_stay_frames = int(min_stay_at_reward_s * frame_rate)  # 2 sec
    
    enter_reward_zone = 0
    consecutive_count = 0
    first_entry_latency = len(individual_x_trim) / frame_rate
    first_entry_frame_on_filtered = None
    time_spent_in_zone = 0
    crossings_per_m = 0

    for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
        if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
            time_spent_in_zone += 1
            if first_entry_frame_on_filtered is None:
                first_entry_frame_on_filtered = i
            if consecutive_count == 0:
                enter_reward_zone += 1
                consecutive_count = 1
        else:
            consecutive_count = 0

    if first_entry_frame_on_filtered is not None:
        first_entry_frameX = np.where(individual_x == individual_x_filt[int(first_entry_frame_on_filtered)])
        first_entry_frameY = np.where(individual_y == individual_y_filt[int(first_entry_frame_on_filtered)])
        common = np.intersect1d(first_entry_frameX, first_entry_frameY)
        first_entry_frame = int(common[0])
        first_entry_latency = (first_entry_frame - start_frame) / frame_rate
        crossings_per_m = round(enter_reward_zone / (round(total_distance) / 100), 2)
        time_spent_in_zone = time_spent_in_zone / frame_rate

    # ========== Compute Permuted metrics ==========
    np.random.seed(seed)

    random_time_spent_in_zone = []
    random_first_entry_latency = []
    random_crossings_per_m = []
    random_targets = []

    while len(random_time_spent_in_zone) < n_permutations:
        r = table_radius * np.sqrt(np.random.rand())
        theta = 2 * np.pi * np.random.rand()
        rx = table_center_x + r * np.cos(theta)
        ry = table_center_y + r * np.sin(theta)

        if calculate_relative_distance(rx, ry, reward_x, reward_y) <= reward_zone:
            continue
        if calculate_relative_distance(rx, ry, x_start, y_start) <= start_zone:
            continue

        rand_enter_reward_zone = 0
        consecutive_count = 0
        rand_first_entry_latency = len(individual_x_trim) / frame_rate
        rand_first_entry_frame_on_filtered = None
        rand_time_spent_in_zone = 0
        rand_crossings_per_m = 0

        for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
            if calculate_relative_distance(x, y, rx, ry) <= reward_zone:
                rand_time_spent_in_zone += 1
                if rand_first_entry_frame_on_filtered is None:
                    rand_first_entry_frame_on_filtered = i
                if consecutive_count == 0:
                    rand_enter_reward_zone += 1
                    consecutive_count = 1
            else:
                consecutive_count = 0

        if rand_first_entry_frame_on_filtered is not None:
            rand_first_entry_frameX = np.where(individual_x == individual_x_filt[int(rand_first_entry_frame_on_filtered)])
            rand_first_entry_frameY = np.where(individual_y == individual_y_filt[int(rand_first_entry_frame_on_filtered)])
            common = np.intersect1d(rand_first_entry_frameX, rand_first_entry_frameY)
            rand_first_entry_frame = int(common[0])
            rand_first_entry_latency = (rand_first_entry_frame - start_frame) / frame_rate
            rand_crossings_per_m = round(rand_enter_reward_zone / (round(total_distance) / 100), 2)
            rand_time_spent_in_zone = rand_time_spent_in_zone / frame_rate

        random_targets.append([rx, ry])
        random_first_entry_latency.append(rand_first_entry_latency)
        random_crossings_per_m.append(rand_crossings_per_m)
        random_time_spent_in_zone.append(rand_time_spent_in_zone)

    random_targets = np.array(random_targets)

    # ========== CALCULATE P-VALUES ==========
    p_value_time_spent_in_zone = (np.sum(random_time_spent_in_zone >= np.array(time_spent_in_zone)) + 1) / (n_permutations + 1)
    p_value_crossings_per_m = (np.sum(random_crossings_per_m >= np.array(crossings_per_m)) + 1) / (n_permutations + 1)
    p_value_first_entry_latency = (np.sum(random_first_entry_latency <= np.array(first_entry_latency)) + 1) / (n_permutations + 1)

    # ========== COMBINED PERFORMANCE ==========
    mean_time_spent_in_zone = np.mean(random_time_spent_in_zone)
    std_time_spent_in_zone = np.std(random_time_spent_in_zone)
    mean_crossings_per_m = np.mean(random_crossings_per_m)
    std_crossings_per_m = np.std(random_crossings_per_m)
    mean_first_entry_latency = np.mean(random_first_entry_latency)
    std_first_entry_latency = np.std(random_first_entry_latency)

    z_time_obs = (time_spent_in_zone - mean_time_spent_in_zone) / std_time_spent_in_zone if std_time_spent_in_zone > 0 else 0
    z_crossing_obs = (crossings_per_m - mean_crossings_per_m) / std_crossings_per_m if std_crossings_per_m > 0 else 0
    z_latency_obs = -(first_entry_latency - mean_first_entry_latency) / std_first_entry_latency if std_first_entry_latency > 0 else 0


    observed_perf_3m = z_time_obs + z_crossing_obs + z_latency_obs
    observed_perf_2m = z_time_obs + z_crossing_obs

    z_time_spent_in_zone_perm = (random_time_spent_in_zone - mean_time_spent_in_zone) / std_time_spent_in_zone if std_time_spent_in_zone > 0 else np.zeros_like(random_time_spent_in_zone)
    z_crossings_perm = (random_crossings_per_m - mean_crossings_per_m) / std_crossings_per_m if std_crossings_per_m > 0 else np.zeros_like(random_crossings_per_m)
    z_latency_perm = -(random_first_entry_latency - mean_first_entry_latency) / std_first_entry_latency if std_first_entry_latency > 0 else np.zeros_like(random_first_entry_latency)

    random_perf_3m = z_time_spent_in_zone_perm + z_crossings_perm + z_latency_perm
    random_perf_2m = z_time_spent_in_zone_perm + z_crossings_perm

    p_value_perf_3m = (np.sum(random_perf_3m >= observed_perf_3m) + 1) / (n_permutations + 1)
    p_value_perf_2m = (np.sum(random_perf_2m >= observed_perf_2m) + 1) / (n_permutations + 1)

    return {
        "p_value_perf_3m": p_value_perf_3m,
        "p_value_perf_2m": p_value_perf_2m,
        "p_value_time_spent_in_zone": p_value_time_spent_in_zone,
        "p_value_crossings_per_m": p_value_crossings_per_m,
        "p_value_first_entry_latency": p_value_first_entry_latency,
        "z_time_obs": z_time_obs,
        "z_crossing_obs": z_crossing_obs,
        "z_latency_obs": z_latency_obs,
    }

def sholl_analysis(x, y, xr, yr, x_mask, y_mask, mask_radius=None, max_radius=100, step=10, buffer=20):
    if mask_radius is not None:
        x, y = apply_circular_mask(x, y, x_mask, y_mask, mask_radius)

    path_coords = np.column_stack((x, y))
    target = np.array([xr, yr])
    dist_to_target = distance.cdist(path_coords, [target]).flatten()

    radii = np.arange(step, max_radius + step, step)
    intersections = np.zeros(len(radii), dtype=int)

    for i, radius in enumerate(radii):
        crossings = np.where((dist_to_target[:-1] < radius) & (dist_to_target[1:] >= radius))[0]
        intersections[i] = len(crossings)

    radii_cm = radii / pixel_to_cm  # Convert to cm
    norm_intersections = intersections / (2 * np.pi * radii_cm)

    return norm_intersections, radii


In [4]:
##############################################################################
# Reset (pour pouvoir relancer cette cellule sans créer de doublons)
##############################################################################
counter = 0
counterProbe = 0
previousmice = 0
previous_session_type = None
Summary_table = pd.DataFrame()
Sholl_table = pd.DataFrame()

##############################################################################
# Pre-pass: build day numbering from date folders
##############################################################################
import re
from datetime import datetime

session_map = {"Training": "TD", "Test": "P", "Habituation": "HD", "Post": "Post"}

day_mapping = {}  # {(mice, session_type): {date_str: day_number}}

for h5_path in Path(folder_path).rglob("*.h5"):
    parts = h5_path.parts
    idx = parts.index("Cheeseboard")

    mice_tmp           = parts[idx + 2]
    session_folder_tmp = parts[idx + 4]
    date_str_tmp        = parts[idx + 5]
    session_type_tmp    = session_map.get(session_folder_tmp, session_folder_tmp)

    key = (mice_tmp, session_type_tmp)
    day_mapping.setdefault(key, set()).add(date_str_tmp)

for key, dates in day_mapping.items():
    sorted_dates = sorted(dates, key=lambda d: datetime.strptime(d, "%d_%m_%Y"))
    day_mapping[key] = {date: i + 1 for i, date in enumerate(sorted_dates)}


##############################################################################
# Load position files (fichiers uniques, communs à tout le dataset)
##############################################################################
cheeseboard_root = Path(folder_path)

with open(cheeseboard_root / "Reward_position.txt", "r") as file:
    text = file.read()
numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
reward_x, reward_y = map(float, numbers)

with open(cheeseboard_root / "Center_position.txt", "r") as file:
    text = file.read()
numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text)
table_center_x, table_center_y = map(float, numbers)


##############################################################################
# Filter duplicate snapshots : keep only ONE .h5 per video
##############################################################################
# Plusieurs snapshots (ex: snapshot_010, snapshot_030, snapshot_160) peuvent
# exister pour une même vidéo -> on privilégie snapshot_010 (le modèle
# standard utilisé pour la quasi-totalité du dataset), et si absent pour une
# vidéo donnée, on prend le snapshot le plus élevé disponible en repli.

PREFERRED_SNAPSHOT = "010"

def get_snapshot_number(h5_path):
    match = re.search(r'snapshot_(\d+)', h5_path.name)
    return match.group(1) if match else None

def select_h5(files):
    preferred = [f for f in files if get_snapshot_number(f) == PREFERRED_SNAPSHOT]
    if preferred:
        return preferred[0]
    return max(files, key=lambda f: int(get_snapshot_number(f) or -1))

all_h5 = list(Path(folder_path).rglob("*.h5"))
print(f"Nombre total de fichiers .h5 trouvés : {len(all_h5)}")

seen_videos = {}
for h5_path in all_h5:
    video_key = (h5_path.parent, h5_path.stem.split("DLC")[0])
    seen_videos.setdefault(video_key, []).append(h5_path)

duplicates_report = {k: v for k, v in seen_videos.items() if len(v) > 1}
print(f"Nombre de vidéos avec plusieurs snapshots : {len(duplicates_report)}")

h5_to_process = [select_h5(files) for files in seen_videos.values()]
print(f"Nombre de fichiers .h5 à traiter après sélection du snapshot : {len(h5_to_process)}")

fallback_used = [f for f in h5_to_process if get_snapshot_number(f) != PREFERRED_SNAPSHOT]
print(f"Vidéos où {PREFERRED_SNAPSHOT} n'était pas disponible ({len(fallback_used)}) :")
for f in fallback_used:
    print(f"  {f}")


##############################################################################
# Loop through all .h5 files in the folder and process them
##############################################################################

errors_log = []  # (filename, error_type, error_message)

for filename in h5_to_process:

    filename = os.path.normpath(str(filename))

    try:
        print(filename)

        ########################
        # Extract metadata from folder path
        ########################
        p = Path(filename)
        parts = p.parts
        idx = parts.index("Cheeseboard")

        genotype       = parts[idx + 1]   # "APPPS1" ou "WT"
        mice           = parts[idx + 2]   # ex: "AHAD01.37"
        age            = parts[idx + 3]   # ex: "2months"
        session_folder = parts[idx + 4]   # "Training", "Test", "Habituation", "Post"
        date_str       = parts[idx + 5]   # ex: "10_07_2026"

        session_type = session_map.get(session_folder, session_folder)
        day = day_mapping.get((mice, session_type), {}).get(date_str, np.nan)

        trial_match = re.search(r'-(\d+)(?=DLC|\.)', p.name)
        trial = trial_match.group(1) if trial_match else '1'

        ########################
        # Define parameters
        # reward_zone dépend du type de session (8 cm TD, 20 cm sinon)
        ########################

        table_radius = 606 / 2
        pixel_to_cm = table_radius / 60  # table is 60 cm radius in real life
        reward_zone     = 8 * pixel_to_cm if 'TD' in session_type else 20 * pixel_to_cm
        start_zone      = 30 * pixel_to_cm  # 30 cm start zone in pixels
        min_stay_at_reward_s = 2             # seconds
        table_margin    = 5 * pixel_to_cm   # 5 cm margin to account for head pitch
        nose_poke_zone = 5 * pixel_to_cm

        ### Load HDF5 file ###
        df = pd.read_hdf(filename)
        directory = os.path.dirname(filename)
        timestamps_path = Path(directory, 'timeStamps.csv')
        if timestamps_path.exists():
            timestamps = pd.read_csv(timestamps_path)
            frame_rate = round(1 / (np.mean(np.diff(timestamps.iloc[:, 1])) / 1000))
        else:
            frame_rate = 16  # fps /!\ CHANGE ACCORDING TO YOUR DATA

        min_stay_at_reward = min_stay_at_reward_s * frame_rate

        ### Clean the signal ###
        df.iloc[:, 0] = df.apply(lambda row: row.iloc[0] if row.iloc[2] > 0.5 else np.nan, axis=1)
        df.iloc[:, 1] = df.apply(lambda row: row.iloc[1] if row.iloc[2] > 0.5 else np.nan, axis=1)

        X = df.iloc[:, 0]
        Y = df.iloc[:, 1]

        individual_xO = np.array(X.values)
        individual_yO = np.array(Y.values)

        # Keep only points on the cheeseboard
        for i, x in enumerate(individual_xO):
            y = individual_yO[i]
            if calculate_relative_distance(x, y, table_center_x, table_center_y) >= table_radius:
                individual_xO[i] = np.nan
                individual_yO[i] = np.nan

        individual_xOO = remove_short_sequences(individual_xO, max_len=3)
        individual_yOO = remove_short_sequences(individual_yO, max_len=3)

        long_seq_x = find_long_non_nan_sequences(individual_xOO)
        long_seq_y = find_long_non_nan_sequences(individual_yOO)
        if len(long_seq_x) == 0 or len(long_seq_y) == 0:
            raise ValueError("Aucune séquence de tracking assez longue trouvée (find_long_non_nan_sequences vide)")

        x_start = long_seq_x[0][0]
        y_start = long_seq_y[0][0]

        start_frame = np.where(individual_xOO == x_start)[0][0].item()
        individual_xOO[:start_frame] = np.nan
        individual_yOO[:start_frame] = np.nan

        individual_x1, individual_y1 = replace_high_speed_points_with_nan(individual_xOO, individual_yOO, speed_threshold=10)

        last_frame = None
        for i in range(len(individual_x1)-1, 0, -1):
            if not np.isnan(individual_x1[i]) and not np.isnan(individual_x1[i-1]):
                last_frame = i
                break
        if last_frame is None:
            raise ValueError("Impossible de trouver last_frame (pas assez de points valides consécutifs)")

        individual_x2, individual_y2 = interpolate_2d_path(individual_x1[start_frame:last_frame], individual_y1[start_frame:last_frame], kind='nearest')
        individual_x3, individual_y3 = limit_speed(individual_x2, individual_y2, max_speed=20)

        individual_x = np.concatenate((individual_x1[:start_frame], individual_x3))
        individual_y = np.concatenate((individual_y1[:start_frame], individual_y3))

        ##############################################################################
        # Filter immobility and thigmotaxia
        ##############################################################################

        border = 8 * pixel_to_cm
        border_inner = table_radius - border

        # Immobility detection : stays within 5cm radius for 3 consecutive seconds
        immobility_radius_px = 2.5 * pixel_to_cm  # 5cm diameter = 2.5cm radius
        min_stay_frames = int(frame_rate * 3)  # 3 seconds

        immobility_mask = np.zeros(len(individual_x), dtype=bool)

        i = 0
        while i < len(individual_x) - min_stay_frames:
            if np.isnan(individual_x[i]):
                i += 1
                continue

            cx, cy = individual_x[i], individual_y[i]  # center of the zone

            # Check if all frames in the window stay within the radius
            window_x = individual_x[i:i + min_stay_frames]
            window_y = individual_y[i:i + min_stay_frames]

            distances = np.sqrt((window_x - cx)**2 + (window_y - cy)**2)

            if np.all(np.where(np.isnan(distances), False, distances <= immobility_radius_px)):
                # Mark all frames in this immobility bout as immobile
                j = i + min_stay_frames
                while j < len(individual_x):
                    if np.isnan(individual_x[j]):
                        break
                    d = np.sqrt((individual_x[j] - cx)**2 + (individual_y[j] - cy)**2)
                    if d <= immobility_radius_px:
                        j += 1
                    else:
                        break
                immobility_mask[i:j] = True
                i = j  # jump to end of immobility bout
            else:
                i += 1

        print(f'Frames excluded - immobility: {immobility_mask.sum()/len(immobility_mask)*100:.1f}%')

        # Mask thigmotaxia
        distance_from_center = np.array([
            calculate_relative_distance(x, y, table_center_x, table_center_y)
            for x, y in zip(individual_x, individual_y)
        ])
        thigmotaxia_mask = distance_from_center >= border_inner

        # Combined mask
        exclude_mask = immobility_mask | thigmotaxia_mask

        # Apply directly
        individual_x[exclude_mask] = np.nan
        individual_y[exclude_mask] = np.nan

        print(f'Frames excluded - immobility: {immobility_mask.sum()/len(immobility_mask)*100:.1f}%')
        print(f'Frames excluded - thigmotaxia: {thigmotaxia_mask.sum()/len(thigmotaxia_mask)*100:.1f}%')
        print(f'Frames excluded - total: {exclude_mask.sum()/len(exclude_mask)*100:.1f}%')


        ########################
        # Compute metrics
        ########################

        start_quadrant = get_quadrant(x_start, y_start, table_center_x, table_center_y)
        reward_quadrant = get_quadrant(reward_x, reward_y, table_center_x, table_center_y)

        if timestamps_path.exists():
            start_time = timestamps.iloc[start_frame, 1].item() / 1000
            end_time = timestamps.iloc[-1, 1].item() / 1000
            duration_trial = end_time - start_time
        else:
            start_time = start_frame / frame_rate
            end_time = last_frame / frame_rate
            duration_trial = (last_frame - start_frame) / frame_rate

        total_distance, distances = calculate_distance_run(individual_x[start_frame:last_frame], individual_y[start_frame:last_frame])
        speed = np.nanmean(distances) / pixel_to_cm * frame_rate

        # Remove NaN values (jointly for x and y)
        valid_mask = ~np.isnan(individual_x) & ~np.isnan(individual_y)
        individual_x_filt = individual_x[valid_mask]
        individual_y_filt = individual_y[valid_mask]

        # Time spent at table border (8 cm band)
        border = 8 * pixel_to_cm
        border_inner = table_radius - border
        border_outer = table_radius + table_margin
        time_spent_at_border = 0
        for x, y in zip(individual_x_filt, individual_y_filt):
            dist = calculate_relative_distance(x, y, table_center_x, table_center_y)
            if border_inner <= dist <= border_outer:
                time_spent_at_border += 1
        time_spent_at_border_sec = time_spent_at_border / frame_rate

        # ========================
        # FIRST ENTRY to reward zone (simple: first frame inside the zone)
        # ========================
        latency_first_entry = np.nan
        distance_first_entry = np.nan
        distances_first_entry = None
        first_entry_frame_on_filt = np.nan

        for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
            if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                first_entry_frame_on_filt = i
                break

        if not np.isnan(first_entry_frame_on_filt):
            frame_x = np.where(individual_x == individual_x_filt[int(first_entry_frame_on_filt)])[0]
            frame_y = np.where(individual_y == individual_y_filt[int(first_entry_frame_on_filt)])[0]
            common_frame = np.intersect1d(frame_x, frame_y)
            if len(common_frame) > 0:
                first_entry_frame = int(common_frame[0])
                if timestamps_path.exists():
                    first_entry_time = timestamps.iloc[first_entry_frame, 1].item() / 1000
                    latency_first_entry = first_entry_time - start_time
                else:
                    latency_first_entry = (first_entry_frame - start_frame) / frame_rate
                distance_first_entry, distances_first_entry = calculate_distance_run(
                    individual_x[start_frame:first_entry_frame],
                    individual_y[start_frame:first_entry_frame]
                )

        # ========================
        # MIN STAY: first frame where mouse stayed >= 2s continuously in zone
        # ========================
        latency_min_stay = np.nan
        distance_min_stay = np.nan
        distances_min_stay = None
        found_reward_frame_on_filtered = np.nan
        found_reward_time = np.nan
        enter_reward_zone_minstay = 0
        consecutive_count = 0
        timespent = 0
        reward_found = False

        for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
            if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                timespent += 1
                if consecutive_count == 0:
                    enter_reward_zone_minstay += 1
                    consecutive_count = 1
            else:
                consecutive_count = 0
                timespent = 0
            if timespent > min_stay_at_reward:
                found_reward_frame_on_filtered = (i - min_stay_at_reward)
                reward_found = True
                break

        # Fallback if mouse was in zone but not long enough
        if not reward_found and len(X) / frame_rate < 250:
            print(f'Reward not detected long enough — using max dwell for {mice}, {session_type}, trial {trial}')
            All_timespent = pd.DataFrame(columns=['timespent_in_zone', 'frame_i'])
            enter_reward_zone_minstay = 0
            consecutive_count = 0
            timespent = 0
            for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
                if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                    timespent += 1
                    if consecutive_count == 0:
                        enter_reward_zone_minstay += 1
                        consecutive_count = 1
                    All_timespent.loc[enter_reward_zone_minstay, 'timespent_in_zone'] = timespent
                    All_timespent.loc[enter_reward_zone_minstay, 'frame_i'] = i - timespent
                else:
                    consecutive_count = 0
                    timespent = 0
            if All_timespent.empty:
                print(f"  {mice}, {session_type}, trial {trial} → never entered reward zone")
                found_reward_frame_on_filtered = np.nan
            else:
                found_reward_frame_on_filtered = int(All_timespent.loc[All_timespent['timespent_in_zone'].idxmax(), 'frame_i'])
                reward_found = True

        if reward_found and not np.isnan(found_reward_frame_on_filtered):
            found_reward_frame1 = np.where(individual_x == individual_x_filt[int(found_reward_frame_on_filtered)])
            found_reward_frame2 = np.where(individual_y == individual_y_filt[int(found_reward_frame_on_filtered)])
            common = np.intersect1d(found_reward_frame1, found_reward_frame2)
            found_reward_frame = int(common[0])
            if timestamps_path.exists():
                found_reward_time = timestamps.iloc[found_reward_frame, 1].item() / 1000
                latency_min_stay = found_reward_time - start_time
            else:
                found_reward_time = None
                latency_min_stay = (found_reward_frame - start_frame) / frame_rate
            found_reward_frame_int = int(found_reward_frame)
            distance_min_stay, distances_min_stay = calculate_distance_run(
                individual_x[start_frame:found_reward_frame_int],
                individual_y[start_frame:found_reward_frame_int]
            )

        # Comptage séparé des crossings sur TOUT le trial (sans break)
        enter_reward_zone = 0
        consecutive_count = 0
        for x, y in zip(individual_x_filt, individual_y_filt):
            if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                if consecutive_count == 0:
                    enter_reward_zone += 1
                    consecutive_count = 1
            else:
                consecutive_count = 0

        crossings_per_m = round(enter_reward_zone / (round(total_distance) / 100), 2) if total_distance > 0 else 0

        # Total time spent in reward zone (entire trial)
        time_spent_in_zone = 0
        for x, y in zip(individual_x_filt, individual_y_filt):
            if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                time_spent_in_zone += 1
        time_spent_in_zone = time_spent_in_zone / frame_rate

        # Time spent in each quadrant (Probe / HD only)
        if session_type != "TD":
            time_spent_south = 0
            time_spent_west = 0
            time_spent_north = 0
            time_spent_east = 0
            for x, y in zip(individual_x_filt, individual_y_filt):
                q = get_quadrant(x, y, table_center_x, table_center_y)
                if q == "South":
                    time_spent_south += 1
                elif q == "West":
                    time_spent_west += 1
                elif q == "North":
                    time_spent_north += 1
                elif q == "East":
                    time_spent_east += 1
            time_spent_south_sec = time_spent_south / frame_rate
            time_spent_west_sec  = time_spent_west  / frame_rate
            time_spent_north_sec = time_spent_north / frame_rate
            time_spent_east_sec  = time_spent_east  / frame_rate

        ########################
        # Nose poke (Probe / HD only)
        ########################
        if session_type != "TD":
            nose_poke_in_well = False
            for i, (x, y) in enumerate(zip(individual_x_filt, individual_y_filt)):
                if calculate_relative_distance(x, y, reward_x, reward_y) <= nose_poke_zone:
                    nose_poke_in_well = True
                    nose_poke_in_well_frame = i
                    break

            if nose_poke_in_well:
                first_nosepoke_frameX = np.where(individual_x == individual_x_filt[int(nose_poke_in_well_frame)])
                first_nosepoke_frameY = np.where(individual_y == individual_y_filt[int(nose_poke_in_well_frame)])
                common = np.intersect1d(first_nosepoke_frameX, first_nosepoke_frameY)
                first_nosepoke_frame = int(common[0])
                first_nosepoke_latency = (first_nosepoke_frame - start_frame) / frame_rate

                # Time in zone and crossings after nose poke
                enter_rz_after_np = 0
                consecutive_count = 0
                np_time_spent_in_zone = 0
                for x, y in zip(individual_x_filt[nose_poke_in_well_frame:], individual_y_filt[nose_poke_in_well_frame:]):
                    if calculate_relative_distance(x, y, reward_x, reward_y) <= reward_zone:
                        np_time_spent_in_zone += 1
                        if consecutive_count == 0:
                            enter_rz_after_np += 1
                            consecutive_count = 1
                    else:
                        consecutive_count = 0
                distance_after_np, _ = calculate_distance_run(
                    individual_x[nose_poke_in_well_frame:], individual_y[nose_poke_in_well_frame:]
                )
                crossings_per_m_after_np = round(enter_rz_after_np / (round(distance_after_np) / 100), 2)
                time_in_zone_after_np = np_time_spent_in_zone / frame_rate

                # Border time before / after nose poke
                time_at_border_before_np = sum(
                    1 for x, y in zip(individual_x_filt[:nose_poke_in_well_frame], individual_y_filt[:nose_poke_in_well_frame])
                    if border_inner <= calculate_relative_distance(x, y, table_center_x, table_center_y) <= border_outer
                )
                time_at_border_after_np = sum(
                    1 for x, y in zip(individual_x_filt[nose_poke_in_well_frame:], individual_y_filt[nose_poke_in_well_frame:])
                    if border_inner <= calculate_relative_distance(x, y, table_center_x, table_center_y) <= border_outer
                )
                time_at_border_before_np_s = time_at_border_before_np / frame_rate
                time_at_border_after_np_s  = time_at_border_after_np  / frame_rate

        ########################
        # Permutation tests + Sholl  (Probe / HD only)
        # 3 fenêtres temporelles : 2 min, 3 dernières min (min 2→3), 3 min complètes
        ########################
        if session_type != "TD":
            trial_duration_frames = last_frame - start_frame

            # Window 1 : 2 first minutes
            t1 = 60 * 2
            end1 = int(start_frame + (t1 * frame_rate))
            results_2min = compute_performance_metrics(
                individual_x=individual_x, individual_y=individual_y,
                start_frame=start_frame, t=t1, frame_rate=frame_rate,
                reward_x=reward_x, reward_y=reward_y, reward_zone=reward_zone,
                table_radius=table_radius, table_center_x=table_center_x, table_center_y=table_center_y,
                x_start=x_start, y_start=y_start, start_zone=start_zone,
                total_distance=total_distance, calculate_relative_distance=calculate_relative_distance,
                n_permutations=1000, seed=42
            )
            norm_sholl_2min, radii_2min = sholl_analysis(
                individual_x[start_frame:end1], individual_y[start_frame:end1],
                reward_x, reward_y, table_center_x, table_center_y,
                mask_radius=table_radius, max_radius=table_radius * 2, step=10
            )

            # Window 2 : last minute (min 2 → 3)
            t2_start = 60 * 2
            t2_end   = 60 * 3
            start2 = int(start_frame + (t2_start * frame_rate))
            end2   = int(start_frame + (t2_end   * frame_rate))
            x_last3 = individual_x[start2:end2]
            y_last3 = individual_y[start2:end2]
            results_last3min = compute_performance_metrics(
                individual_x=x_last3, individual_y=y_last3,
                start_frame=0, t=(t2_end - t2_start), frame_rate=frame_rate,
                reward_x=reward_x, reward_y=reward_y, reward_zone=reward_zone,
                table_radius=table_radius, table_center_x=table_center_x, table_center_y=table_center_y,
                x_start=x_start, y_start=y_start, start_zone=start_zone,
                total_distance=total_distance, calculate_relative_distance=calculate_relative_distance,
                n_permutations=1000, seed=42
            )
            norm_sholl_last3min, radii_last3min = sholl_analysis(
                x_last3, y_last3,
                reward_x, reward_y, table_center_x, table_center_y,
                mask_radius=table_radius, max_radius=table_radius * 2, step=10
            )

            # Window 3 : full 3 minutes
            t3 = 60 * 3
            end3 = int(start_frame + (t3 * frame_rate))
            results_5min = compute_performance_metrics(
                individual_x=individual_x, individual_y=individual_y,
                start_frame=start_frame, t=t3, frame_rate=frame_rate,
                reward_x=reward_x, reward_y=reward_y, reward_zone=reward_zone,
                table_radius=table_radius, table_center_x=table_center_x, table_center_y=table_center_y,
                x_start=x_start, y_start=y_start, start_zone=start_zone,
                total_distance=total_distance, calculate_relative_distance=calculate_relative_distance,
                n_permutations=1000, seed=42
            )
            norm_sholl_5min, radii_5min = sholl_analysis(
                individual_x[start_frame:end3], individual_y[start_frame:end3],
                reward_x, reward_y, table_center_x, table_center_y,
                mask_radius=table_radius, max_radius=table_radius * 2, step=10
            )

        ########################
        # Create summary table
        ########################

        # --- Identifiers ---
        Summary_table.loc[counter, 'mice']         = mice
        Summary_table.loc[counter, 'genotype']     = genotype
        Summary_table.loc[counter, 'age']          = age
        Summary_table.loc[counter, 'session_type'] = session_type
        Summary_table.loc[counter, 'session']      = day
        Summary_table.loc[counter, 'trial']        = int(trial) if str(trial).isdigit() else trial
        Summary_table.loc[counter, 'start_time']   = start_time
        Summary_table.loc[counter, 'end_time']     = end_time

        Summary_table.loc[counter, 'start_quadrant']  = start_quadrant
        Summary_table.loc[counter, 'reward_quadrant'] = reward_quadrant

        # --- General metrics ---
        Summary_table.loc[counter, 'reward_location_pix']      = str([reward_x, reward_y])
        Summary_table.loc[counter, 'reward_zone_radius_cm']    = reward_zone / pixel_to_cm
        Summary_table.loc[counter, 'nosepoke_zone_radius_cm']  = nose_poke_zone / pixel_to_cm
        Summary_table.loc[counter, 'duration_trial_s']         = round(duration_trial, 2)
        Summary_table.loc[counter, 'total_distance_cm']        = round(total_distance, 2)
        Summary_table.loc[counter, 'average_speed_cm_s']       = round(speed, 2)
        Summary_table.loc[counter, 'time_spent_at_border_s']   = round(time_spent_at_border_sec, 2)
        Summary_table.loc[counter, 'time_spent_at_border_percent'] = round(time_spent_at_border_sec / duration_trial * 100, 2)

        # --- Reward zone global ---
        Summary_table.loc[counter, 'time_spent_in_reward_zone_s']       = round(time_spent_in_zone, 2)
        Summary_table.loc[counter, 'time_spent_in_reward_zone_percent']  = round(time_spent_in_zone / duration_trial * 100, 2)
        Summary_table.loc[counter, 'crossings']      = enter_reward_zone
        Summary_table.loc[counter, 'crossings_per_m'] = round(crossings_per_m, 2)

        # --- First entry metrics ---
        Summary_table.loc[counter, 'latency_first_entry_s']    = round(latency_first_entry, 2) if not np.isnan(latency_first_entry) else np.nan
        Summary_table.loc[counter, 'distance_first_entry_cm']  = round(distance_first_entry, 2) if not np.isnan(distance_first_entry) else np.nan
        if distances_first_entry is not None:
            Summary_table.loc[counter, 'speed_first_entry_cm_s'] = round(np.nanmean(distances_first_entry) / pixel_to_cm * frame_rate, 2)
        else:
            Summary_table.loc[counter, 'speed_first_entry_cm_s'] = np.nan

        # --- Min stay metrics (2s continuous) ---
        Summary_table.loc[counter, 'latency_min_stay_s']    = round(latency_min_stay, 2) if not np.isnan(latency_min_stay) else np.nan
        Summary_table.loc[counter, 'distance_min_stay_cm']  = round(distance_min_stay, 2) if not np.isnan(distance_min_stay) else np.nan
        if distances_min_stay is not None:
            Summary_table.loc[counter, 'speed_min_stay_cm_s'] = round(np.nanmean(distances_min_stay) / pixel_to_cm * frame_rate, 2)
        else:
            Summary_table.loc[counter, 'speed_min_stay_cm_s'] = np.nan

        Summary_table.loc[counter, 'found_reward_time'] = found_reward_time

        # --- Probe / HD specific ---
        if session_type != "TD":

            # Quadrant time
            Summary_table.loc[counter, 'time_in_S_percent'] = round(time_spent_south_sec / duration_trial * 100, 2)
            Summary_table.loc[counter, 'time_in_W_percent'] = round(time_spent_west_sec  / duration_trial * 100, 2)
            Summary_table.loc[counter, 'time_in_N_percent'] = round(time_spent_north_sec / duration_trial * 100, 2)
            Summary_table.loc[counter, 'time_in_E_percent'] = round(time_spent_east_sec  / duration_trial * 100, 2)

            # Nose poke
            Summary_table.loc[counter, 'nose_poke_in_well'] = str(nose_poke_in_well)
            if nose_poke_in_well:
                Summary_table.loc[counter, 'first_nosepoke_latency_s']        = round(first_nosepoke_latency, 2)
                Summary_table.loc[counter, 'before_np_time_at_border_percent'] = round(time_at_border_before_np_s / first_nosepoke_latency * 100, 2)
                Summary_table.loc[counter, 'after_np_time_at_border_percent']  = round(time_at_border_after_np_s  / (duration_trial - first_nosepoke_latency) * 100, 2)
                Summary_table.loc[counter, 'after_np_time_in_zone_percent']    = round(time_in_zone_after_np / (duration_trial - first_nosepoke_latency) * 100, 2)
                Summary_table.loc[counter, 'after_np_crossings_per_m']         = round(crossings_per_m_after_np, 2)

            # p-values sur 3 fenêtres temporelles
            def _pval(res, key):
                v = res[key]
                return round(v, 4) if not np.isnan(v) else np.nan

            # 2 first minutes
            Summary_table.loc[counter, 'z_time_obs_2min']     = round(results_2min['z_time_obs'], 4)
            Summary_table.loc[counter, 'z_crossing_obs_2min'] = round(results_2min['z_crossing_obs'], 4)
            Summary_table.loc[counter, 'z_latency_obs_2min']  = round(results_2min['z_latency_obs'], 4)
            Summary_table.loc[counter, 'p_value_perf3m_2min'] = _pval(results_2min, 'p_value_perf_3m')
            Summary_table.loc[counter, 'p_value_perf2m_2min'] = _pval(results_2min, 'p_value_perf_2m')
            Summary_table.loc[counter, 'p_value_time_2min']   = _pval(results_2min, 'p_value_time_spent_in_zone')
            Summary_table.loc[counter, 'p_value_crossings_2min']    = _pval(results_2min, 'p_value_crossings_per_m')
            Summary_table.loc[counter, 'p_value_first_entry_2min']  = _pval(results_2min, 'p_value_first_entry_latency')

            # Last minute (min 2 → 3)
            Summary_table.loc[counter, 'z_time_obs_last3min']     = round(results_last3min['z_time_obs'], 4)
            Summary_table.loc[counter, 'z_crossing_obs_last3min'] = round(results_last3min['z_crossing_obs'], 4)
            Summary_table.loc[counter, 'z_latency_obs_last3min']  = round(results_last3min['z_latency_obs'], 4)
            Summary_table.loc[counter, 'p_value_perf3m_last3min'] = _pval(results_last3min, 'p_value_perf_3m')
            Summary_table.loc[counter, 'p_value_perf2m_last3min'] = _pval(results_last3min, 'p_value_perf_2m')
            Summary_table.loc[counter, 'p_value_time_last3min']   = _pval(results_last3min, 'p_value_time_spent_in_zone')
            Summary_table.loc[counter, 'p_value_crossings_last3min']   = _pval(results_last3min, 'p_value_crossings_per_m')
            Summary_table.loc[counter, 'p_value_first_entry_last3min'] = _pval(results_last3min, 'p_value_first_entry_latency')

            # Full 3 minutes
            Summary_table.loc[counter, 'z_time_obs_5min']     = round(results_5min['z_time_obs'], 4)
            Summary_table.loc[counter, 'z_crossing_obs_5min'] = round(results_5min['z_crossing_obs'], 4)
            Summary_table.loc[counter, 'z_latency_obs_5min']  = round(results_5min['z_latency_obs'], 4)
            Summary_table.loc[counter, 'p_value_perf3m_5min'] = _pval(results_5min, 'p_value_perf_3m')
            Summary_table.loc[counter, 'p_value_perf2m_5min'] = _pval(results_5min, 'p_value_perf_2m')
            Summary_table.loc[counter, 'p_value_time_5min']   = _pval(results_5min, 'p_value_time_spent_in_zone')
            Summary_table.loc[counter, 'p_value_crossings_5min']   = _pval(results_5min, 'p_value_crossings_per_m')
            Summary_table.loc[counter, 'p_value_first_entry_5min'] = _pval(results_5min, 'p_value_first_entry_latency')

            # Sholl sur 3 fenêtres
            Sholl_table.loc[counterProbe, 'mice']         = mice
            Sholl_table.loc[counterProbe, 'genotype']     = genotype
            Sholl_table.loc[counterProbe, 'age']          = age
            Sholl_table.loc[counterProbe, 'session_type'] = session_type
            Sholl_table.loc[counterProbe, 'session']      = day
            Sholl_table.loc[counterProbe, 'trial']        = int(trial) if str(trial).isdigit() else trial

            for i in range(len(radii_2min)):
                Sholl_table.loc[counterProbe, f'norm_intersec_{int(radii_2min[i])}rad_2min'] = norm_sholl_2min[i]
            for i in range(len(radii_last3min)):
                Sholl_table.loc[counterProbe, f'norm_intersec_{int(radii_last3min[i])}rad_last3min'] = norm_sholl_last3min[i]
            for i in range(len(radii_5min)):
                Sholl_table.loc[counterProbe, f'norm_intersec_{int(radii_5min[i])}rad_5min'] = norm_sholl_5min[i]

            counterProbe += 1

        previousmice = mice
        previous_session_type = session_type
        counter += 1

    except Exception as e:
        print(f"  ❌ ERREUR sur {filename} : {type(e).__name__}: {e}")
        errors_log.append((filename, type(e).__name__, str(e)))
        continue

##############################################################################
# Rapport final
##############################################################################
print(f"\n{'='*60}")
print(f"Fichiers traités avec succès : {counter}")
print(f"Fichiers en erreur : {len(errors_log)}")
print(f"{'='*60}")

if errors_log:
    errors_df = pd.DataFrame(errors_log, columns=["filename", "error_type", "error_message"])
    print("\nRépartition des erreurs par type :")
    print(errors_df["error_type"].value_counts())
    errors_df.to_csv(Path(folder_path) / "errors_log.csv", index=False)
    print(f"\nDétail complet sauvegardé dans : {Path(folder_path) / 'errors_log.csv'}")

Nombre total de fichiers .h5 trouvés : 3974
Nombre de vidéos avec plusieurs snapshots : 43
Nombre de fichiers .h5 à traiter après sélection du snapshot : 3922
Vidéos où 010 n'était pas disponible (1) :
  \\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\APPPS1\AHAD02.06\3months\Test\31_01_2025\AHAD02.06DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_160.h5
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\APPPS1\AHAD01.37\10months\Post\06_11_2025\AHAD01.37-AfterTest9-1DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 61.4%
Frames excluded - immobility: 61.4%
Frames excluded - thigmotaxia: 4.3%
Frames excluded - total: 65.7%


C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\3450040921.py:665: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Sholl_table.loc[counterProbe, f'norm_intersec_{int(radii_last3min[i])}rad_last3min'] = norm_sholl_last3min[i]
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\3450040921.py:665: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Sholl_tabl

\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\APPPS1\AHAD01.37\10months\Post\06_11_2025\AHAD01.37-AfterTest9-2DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 14.7%
Frames excluded - immobility: 14.7%
Frames excluded - thigmotaxia: 33.2%
Frames excluded - total: 47.8%
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\APPPS1\AHAD01.37\10months\Post\06_11_2025\AHAD01.37-AfterTest9-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 0.0%
Frames excluded - immobility: 0.0%
Frames excluded - thigmotaxia: 48.8%
Frames excluded - total: 48.8%


C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])


\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\APPPS1\AHAD01.37\10months\Test\06_11_2025\AHAD01.37-Test9DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 59.1%
Frames excluded - immobility: 59.1%
Frames excluded - thigmotaxia: 6.9%
Frames excluded - total: 66.0%
Reward not detected long enough — using max dwell for AHAD01.37, P, trial 1
  AHAD01.37, P, trial 1 → never entered reward zone


...

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  d

Reward not detected long enough — using max dwell for AHAD21.156, TD, trial 7
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-1DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 18.0%
Frames excluded - immobility: 18.0%
Frames excluded - thigmotaxia: 18.7%
Frames excluded - total: 36.7%
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-2DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 14.6%
Frames excluded - immobility: 14.6%
Frames excluded - thigmotaxia: 16.1%
Frames excluded - total: 30.7%
Reward not detected long enough — using max dwell for AHAD21.156, TD, trial 2
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-3DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 0.0%
Frames excluded 

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])


\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-5DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 41.2%
Frames excluded - immobility: 41.2%
Frames excluded - thigmotaxia: 2.3%
Frames excluded - total: 43.5%
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-6DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 37.7%
Frames excluded - immobility: 37.7%
Frames excluded - thigmotaxia: 1.2%
Frames excluded - total: 38.9%
\\10.69.168.1\crnldata\forgetting\Carla\Cheeseboard\WT\AHAD21.156\3months\Training\16_07_2026\AHAD21.156-Training J8-7DLC_Resnet50_CheeseboardFeb6shuffle1_snapshot_010.h5
Frames excluded - immobility: 37.6%
Frames excluded - immobility: 37.6%
Frames excluded - thigmotaxia: 4.4%
Frames excluded - total: 42.0%

Fichiers traités avec succès : 3918
Fichiers en erreur : 4

Répartiti

C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])
C:\Users\AudreyHay\AppData\Local\Temp\ipykernel_8284\1576754356.py:13: RuntimeWarning: Mean of empty slice
  distances[i] = np.nanmean([x for x in neighbors if not np.isnan(x)])


In [5]:
##############################################################################
# Add metadata (Implant, Sex, Virus, Expe)
##############################################################################

metadata_file = r"C:\Users\AudreyHay\Documents\Carla\Visual Code\HayLabAnalysis\python\_MouseID_ImplantColorCode.csv"

metadata = pd.read_csv(metadata_file, sep=";")
metadata.columns = metadata.columns.str.strip()

metadata = metadata[["MouseID", "Implant", "Sex", "Virus", "Expe"]]

def add_metadata(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    # Retire d'éventuelles colonnes de métadonnées déjà présentes
    # (permet de relancer la cellule sans tout relancer avant)
    meta_cols = ["Implant", "Sex", "Virus", "Expe"]
    df = df.drop(columns=[c for c in meta_cols if c in df.columns], errors="ignore")

    # Fusion avec les métadonnées
    df = df.merge(
        metadata,
        left_on="mice",
        right_on="MouseID",
        how="left"
    )

    # Remplacer les valeurs manquantes
    for col in meta_cols:
        df[col] = df[col].fillna("nan")

    # Supprimer MouseID ajouté par la fusion
    df.drop(columns="MouseID", inplace=True)

    # Réorganiser les colonnes
    cols = list(df.columns)
    for c in meta_cols:
        cols.remove(c)
    idx = cols.index("age") + 1
    cols[idx:idx] = meta_cols

    return df[cols]

# Ajouter les métadonnées aux deux tableaux
Summary_table = add_metadata(Summary_table)
Sholl_table = add_metadata(Sholl_table)

##############################################################################
# Save outputs
##############################################################################

filenameOut = f"{folder_path}/Summary_table_filtered_20cm.xlsx"
Summary_table.to_excel(filenameOut, index=False)

filenameOut = f"{folder_path}/Sholl_analysis_table_filtered_20cm.xlsx"
Sholl_table.to_excel(filenameOut, index=False)

print("Done.")

Done.
